# 🛫 Predicción de Pasajeros Aéreos con LSTM
## Electiva Profesional II – Deep Learning | UDEC Facatativá

---

**Dataset:** Airline Passengers Dataset (1949–1960)  
**Modelo:** Long Short-Term Memory (LSTM) – Red Neuronal Recurrente  
**Objetivo:** Predecir la cantidad de pasajeros mensuales usando datos secuenciales históricos

---

### 📋 Estructura del Notebook
1. Instalación e Importación de Librerías
2. Carga y Exploración del Dataset
3. Preprocesamiento y Normalización
4. Creación de Secuencias Temporales
5. División Train / Test
6. Arquitectura del Modelo LSTM
7. Entrenamiento del Modelo
8. Evaluación y Métricas
9. Visualización de Resultados
10. Análisis y Conclusiones

## ⚙️ 1. Instalación e Importación de Librerías

In [ ]:
# ============================================================
# BLOQUE 1: INSTALACIÓN E IMPORTACIÓN DE LIBRERÍAS
# ============================================================
# Se instalan librerías adicionales para visualización profesional
# y manejo de datos.

!pip install plotly --quiet          # Gráficos interactivos
!pip install scikit-learn --quiet    # Métricas y preprocesamiento

# ── Librerías estándar ──────────────────────────────────────
import numpy as np                   # Operaciones numéricas y matriciales
import pandas as pd                  # Manipulación de DataFrames
import matplotlib.pyplot as plt      # Gráficos estáticos
import matplotlib.patches as mpatches
import seaborn as sns                # Visualización estadística
import warnings
warnings.filterwarnings('ignore')    # Silenciar advertencias menores

# ── Deep Learning (TensorFlow / Keras) ─────────────────────
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    LSTM,          # Capa principal: Long Short-Term Memory
    Dense,         # Capa totalmente conectada (salida)
    Dropout,       # Regularización: previene sobreajuste
    BatchNormalization,  # Normaliza las activaciones intermedias
    Input          # Capa de entrada explícita
)
from tensorflow.keras.callbacks import (
    EarlyStopping,      # Detiene el entrenamiento si no mejora
    ReduceLROnPlateau,  # Reduce el learning rate si el modelo estanca
    ModelCheckpoint     # Guarda el mejor modelo durante el entrenamiento
)
from tensorflow.keras.optimizers import Adam  # Optimizador adaptativo

# ── Preprocesamiento y métricas ─────────────────────────────
from sklearn.preprocessing import MinMaxScaler  # Normalización [0, 1]
from sklearn.metrics import (
    mean_squared_error,       # Error cuadrático medio (MSE)
    mean_absolute_error,      # Error absoluto medio (MAE)
    r2_score                  # Coeficiente de determinación R²
)

# ── Gráficos interactivos ───────────────────────────────────
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

# ── Configuración global de estilo ──────────────────────────
plt.style.use('seaborn-v0_8-darkgrid')   # Estilo oscuro profesional
sns.set_palette('husl')                   # Paleta de colores
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

# ── Verificar versiones ─────────────────────────────────────
print('=' * 55)
print('   ENTORNO DE TRABAJO – VERSIONES DE LIBRERÍAS')
print('=' * 55)
print(f'  TensorFlow  : {tf.__version__}')
print(f'  NumPy       : {np.__version__}')
print(f'  Pandas      : {pd.__version__}')
print(f'  Seed global : {RANDOM_SEED}')
print('=' * 55)

## 📦 2. Carga y Exploración del Dataset

> El **Airline Passengers Dataset** contiene el número mensual de pasajeros
> de aerolíneas internacionales entre **enero de 1949** y **diciembre de 1960**.
> Es un clásico benchmark para series temporales con tendencia y estacionalidad.

In [ ]:
# ============================================================
# BLOQUE 2: CARGA Y EXPLORACIÓN DEL DATASET
# ============================================================
# El dataset se descarga directamente desde una URL pública.
# Alternativa: subir el CSV desde Kaggle manualmente.

URL = 'https://raw.githubusercontent.com/jbrownlee/Datasets/master/airline-passengers.csv'

# ── Carga del CSV ────────────────────────────────────────────
df = pd.read_csv(URL, header=0, names=['Mes', 'Pasajeros'])

# ── Conversión de la columna de fecha ───────────────────────
# Se parsea el formato 'YYYY-MM' a datetime para indexar la serie
df['Mes'] = pd.to_datetime(df['Mes'], format='%Y-%m')
df.set_index('Mes', inplace=True)

# ── Vista previa ─────────────────────────────────────────────
print('=' * 55)
print('   VISTA PREVIA DEL DATASET')
print('=' * 55)
print(df.head(12))   # Primer año completo
print()

# ── Información general ──────────────────────────────────────
print('=' * 55)
print('   INFORMACIÓN GENERAL')
print('=' * 55)
print(f'  Registros totales  : {len(df)}')
print(f'  Período            : {df.index.min().date()} → {df.index.max().date()}')
print(f'  Años cubiertos     : {df.index.year.nunique()}')
print(f'  Valores nulos      : {df.isnull().sum().values[0]}')
print()

# ── Estadísticas descriptivas ────────────────────────────────
print('=' * 55)
print('   ESTADÍSTICAS DESCRIPTIVAS')
print('=' * 55)
stats = df.describe().round(2)
print(stats)

In [ ]:
# ============================================================
# BLOQUE 2B: VISUALIZACIÓN EXPLORATORIA INICIAL
# ============================================================
# Se generan 3 paneles para entender la serie antes de modelar.

fig, axes = plt.subplots(3, 1, figsize=(14, 12))
fig.suptitle('Exploración del Dataset – Airline Passengers (1949–1960)',
             fontsize=16, fontweight='bold', y=0.98)

# Panel 1: Serie temporal completa
axes[0].plot(df.index, df['Pasajeros'], color='#2196F3',
             linewidth=2.5, label='Pasajeros mensuales')
axes[0].fill_between(df.index, df['Pasajeros'], alpha=0.15, color='#2196F3')
axes[0].set_title('Serie Temporal Completa', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Miles de pasajeros')
axes[0].legend()
axes[0].set_xlabel('')

# Panel 2: Media móvil (tendencia) vs serie original
rolling_mean = df['Pasajeros'].rolling(window=12).mean()   # Media anual
rolling_std  = df['Pasajeros'].rolling(window=12).std()    # Desv. estándar
axes[1].plot(df.index, df['Pasajeros'], color='#90CAF9',
             linewidth=1.5, alpha=0.8, label='Original')
axes[1].plot(df.index, rolling_mean, color='#E53935',
             linewidth=2.5, label='Media móvil (12m)')
axes[1].fill_between(df.index,
                      rolling_mean - rolling_std,
                      rolling_mean + rolling_std,
                      alpha=0.2, color='#E53935', label='±1 Desv. estándar')
axes[1].set_title('Tendencia y Variabilidad (Media Móvil 12 meses)', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Miles de pasajeros')
axes[1].legend()

# Panel 3: Distribución por mes del año (estacionalidad)
df_temp = df.copy()
df_temp['Mes_num'] = df_temp.index.month
df_temp['Año']     = df_temp.index.year
monthly_avg = df_temp.groupby('Mes_num')['Pasajeros'].mean()
meses_nombres = ['Ene','Feb','Mar','Abr','May','Jun',
                 'Jul','Ago','Sep','Oct','Nov','Dic']
bars = axes[2].bar(meses_nombres, monthly_avg.values,
                    color=sns.color_palette('husl', 12), edgecolor='white', linewidth=0.8)
axes[2].set_title('Promedio de Pasajeros por Mes (Estacionalidad)', fontsize=13, fontweight='bold')
axes[2].set_ylabel('Promedio de pasajeros')
axes[2].set_xlabel('Mes')
# Anotar valores sobre cada barra
for bar, val in zip(bars, monthly_avg.values):
    axes[2].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
                 f'{val:.0f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig('exploracion_dataset.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Gráfico de exploración generado.')

## 🔧 3. Preprocesamiento y Normalización

> Las redes LSTM son sensibles a la escala de los datos.
> Se aplica **MinMaxScaler** para llevar los valores al rango `[0, 1]`.

In [ ]:
# ============================================================
# BLOQUE 3: PREPROCESAMIENTO Y NORMALIZACIÓN
# ============================================================

# ── Extraer valores como array NumPy ────────────────────────
data_raw = df['Pasajeros'].values.astype('float32')
data_raw = data_raw.reshape(-1, 1)   # Shape: (144, 1)

# ── Normalización MinMax: escala los datos a [0, 1] ─────────
# Esto ayuda al gradiente a converger más rápido y evita
# que la LSTM sature sus funciones de activación (tanh, sigmoid).
scaler = MinMaxScaler(feature_range=(0, 1))
data_scaled = scaler.fit_transform(data_raw)

print('=' * 55)
print('   PREPROCESAMIENTO')
print('=' * 55)
print(f'  Shape original  : {data_raw.shape}')
print(f'  Shape escalado  : {data_scaled.shape}')
print(f'  Mín original    : {data_raw.min():.0f} pasajeros')
print(f'  Máx original    : {data_raw.max():.0f} pasajeros')
print(f'  Mín normalizado : {data_scaled.min():.4f}')
print(f'  Máx normalizado : {data_scaled.max():.4f}')
print('=' * 55)
print('✅ Normalización completada.')

## 🔀 4. Creación de Secuencias Temporales

> La LSTM aprende patrones en **ventanas deslizantes** de tiempo.
> Con `LOOK_BACK = 12` usamos los últimos 12 meses para predecir el siguiente.

In [ ]:
# ============================================================
# BLOQUE 4: CREACIÓN DE SECUENCIAS TEMPORALES (VENTANA DESLIZANTE)
# ============================================================
# La función create_sequences convierte la serie 1D en pares (X, y)
# donde X = ventana de 'look_back' pasos y y = valor siguiente.
#
#  Ejemplo con look_back=3:
#  [1, 2, 3] → predice → [4]
#  [2, 3, 4] → predice → [5]  ...

LOOK_BACK = 12   # 12 meses de historia para predecir 1 mes futuro

def create_sequences(data, look_back=12):
    """
    Genera pares de secuencias (X, y) para entrenamiento de RNN/LSTM.

    Parámetros:
    -----------
    data      : array normalizado de forma (n, 1)
    look_back : tamaño de la ventana temporal (número de pasos previos)

    Retorna:
    --------
    X : array de forma (samples, look_back, 1)  ← entrada de la LSTM
    y : array de forma (samples, 1)             ← salida esperada
    """
    X, y = [], []
    for i in range(len(data) - look_back):
        X.append(data[i : i + look_back, 0])    # Ventana de historia
        y.append(data[i + look_back, 0])         # Valor objetivo
    X = np.array(X)
    y = np.array(y)
    # Reshape X a 3D: (muestras, pasos_temporales, características)
    # La LSTM espera siempre 3 dimensiones.
    X = X.reshape(X.shape[0], X.shape[1], 1)
    return X, y

X_all, y_all = create_sequences(data_scaled, LOOK_BACK)

print('=' * 55)
print('   SECUENCIAS GENERADAS')
print('=' * 55)
print(f'  Look-back (ventana)  : {LOOK_BACK} meses')
print(f'  Total de secuencias  : {len(X_all)}')
print(f'  Forma de X           : {X_all.shape}  → (muestras, pasos, features)')
print(f'  Forma de y           : {y_all.shape}  → (muestras,)')
print('=' * 55)
print('✅ Secuencias temporales creadas exitosamente.')

## ✂️ 5. División Train / Test

In [ ]:
# ============================================================
# BLOQUE 5: DIVISIÓN ENTRENAMIENTO / PRUEBA
# ============================================================
# En series temporales NO se mezclan aleatoriamente los datos.
# Se respeta el orden cronológico: primero se entrena, luego se prueba.
# División: 80% train – 20% test

TRAIN_RATIO = 0.80
train_size  = int(len(X_all) * TRAIN_RATIO)

X_train = X_all[:train_size]
X_test  = X_all[train_size:]
y_train = y_all[:train_size]
y_test  = y_all[train_size:]

print('=' * 55)
print('   DIVISIÓN TRAIN / TEST')
print('=' * 55)
print(f'  Total secuencias  : {len(X_all)}')
print(f'  Train (80%)       : {len(X_train)} muestras')
print(f'  Test  (20%)       : {len(X_test)} muestras')
print(f'  Forma X_train     : {X_train.shape}')
print(f'  Forma X_test      : {X_test.shape}')
print('=' * 55)

# ── Diagrama visual de la división ──────────────────────────
fig, ax = plt.subplots(figsize=(14, 3))
serie_idx = df.index[LOOK_BACK:]    # índices correspondientes a las secuencias
ax.plot(serie_idx[:train_size], y_train, color='#1565C0',
        linewidth=2, label=f'Entrenamiento ({train_size} muestras)')
ax.plot(serie_idx[train_size:], y_test,  color='#C62828',
        linewidth=2, label=f'Prueba ({len(y_test)} muestras)')
ax.axvline(serie_idx[train_size], color='gray', linestyle='--', linewidth=1.5, label='Corte train/test')
ax.set_title('División Cronológica – Entrenamiento vs Prueba (datos normalizados)', fontsize=13, fontweight='bold')
ax.set_ylabel('Pasajeros normalizados')
ax.legend()
plt.tight_layout()
plt.show()
print('✅ División completada.')

## 🏗️ 6. Arquitectura del Modelo LSTM

```
Input (12, 1)
    │
    ▼
LSTM (128 unidades, return_sequences=True)
    │
BatchNormalization + Dropout(0.3)
    │
    ▼
LSTM (64 unidades, return_sequences=False)
    │
BatchNormalization + Dropout(0.2)
    │
    ▼
Dense (32, relu)
    │
    ▼
Dense (1) ← Predicción
```

In [ ]:
# ============================================================
# BLOQUE 6: CONSTRUCCIÓN DE LA ARQUITECTURA LSTM
# ============================================================

def build_lstm_model(look_back, lstm1_units=128, lstm2_units=64,
                     dense_units=32, dropout_rate1=0.3,
                     dropout_rate2=0.2, learning_rate=0.001):
    """
    Construye y compila un modelo LSTM de dos capas para regresión.

    Parámetros:
    -----------
    look_back      : número de pasos temporales de entrada
    lstm1_units    : neuronas en la primera capa LSTM (extrae patrones generales)
    lstm2_units    : neuronas en la segunda capa LSTM (refinamiento)
    dense_units    : neuronas en la capa densa intermedia
    dropout_rate1  : tasa de dropout después de LSTM 1 (regularización)
    dropout_rate2  : tasa de dropout después de LSTM 2
    learning_rate  : paso de aprendizaje del optimizador Adam

    Retorna:
    --------
    model : modelo Keras compilado y listo para entrenar
    """
    model = Sequential(name='LSTM_AirlinePassengers')

    # ── Capa de entrada explícita ───────────────────────────
    model.add(Input(shape=(look_back, 1)))

    # ── Primera capa LSTM ───────────────────────────────────
    # return_sequences=True: pasa la secuencia completa a la siguiente LSTM
    model.add(LSTM(units=lstm1_units,
                   return_sequences=True,
                   name='LSTM_1'))
    model.add(BatchNormalization(name='BN_1'))  # Estabiliza el entrenamiento
    model.add(Dropout(rate=dropout_rate1, name='Dropout_1'))  # Previene sobreajuste

    # ── Segunda capa LSTM ───────────────────────────────────
    # return_sequences=False: devuelve solo el último estado oculto
    model.add(LSTM(units=lstm2_units,
                   return_sequences=False,
                   name='LSTM_2'))
    model.add(BatchNormalization(name='BN_2'))
    model.add(Dropout(rate=dropout_rate2, name='Dropout_2'))

    # ── Capa densa de procesamiento ─────────────────────────
    model.add(Dense(units=dense_units, activation='relu', name='Dense_1'))

    # ── Capa de salida (regresión) ──────────────────────────
    # Sin activación → valor continuo (predicción de pasajeros)
    model.add(Dense(units=1, name='Salida'))

    # ── Compilación ─────────────────────────────────────────
    optimizer = Adam(learning_rate=learning_rate)
    model.compile(
        optimizer=optimizer,
        loss='mse',               # Error cuadrático medio como función de pérdida
        metrics=['mae']           # También monitorear MAE durante el entrenamiento
    )
    return model


# ── Crear el modelo ──────────────────────────────────────────
model = build_lstm_model(look_back=LOOK_BACK)

# ── Mostrar resumen de la arquitectura ──────────────────────
print('=' * 55)
print('   ARQUITECTURA DEL MODELO LSTM')
print('=' * 55)
model.summary()
print()

# ── Contar parámetros ────────────────────────────────────────
total_params     = model.count_params()
trainable_params = sum([tf.size(v).numpy() for v in model.trainable_variables])
print(f'  Total de parámetros entrenables: {total_params:,}')

## 🚀 7. Entrenamiento del Modelo

In [ ]:
# ============================================================
# BLOQUE 7: ENTRENAMIENTO DEL MODELO
# ============================================================
# Se definen callbacks para un entrenamiento inteligente:
#   - EarlyStopping   : para si val_loss no mejora en N épocas
#   - ReduceLROnPlateau: baja el lr si el modelo se estanca
#   - ModelCheckpoint : guarda el mejor modelo automáticamente

EPOCHS     = 150
BATCH_SIZE = 16
VAL_SPLIT  = 0.15   # 15% del train como validación

# ── Callbacks ────────────────────────────────────────────────
callbacks = [
    EarlyStopping(
        monitor='val_loss',
        patience=25,           # Espera 25 épocas sin mejora antes de parar
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,            # Reduce lr a la mitad
        patience=10,
        min_lr=1e-6,
        verbose=1
    ),
    ModelCheckpoint(
        filepath='mejor_modelo_lstm.keras',
        monitor='val_loss',
        save_best_only=True,
        verbose=0
    )
]

print('=' * 55)
print('   INICIANDO ENTRENAMIENTO')
print('=' * 55)
print(f'  Épocas máximas  : {EPOCHS}')
print(f'  Batch size      : {BATCH_SIZE}')
print(f'  Validación      : {VAL_SPLIT*100:.0f}% del train')
print('=' * 55)
print()

# ── Entrenamiento ─────────────────────────────────────────────
history = model.fit(
    X_train, y_train,
    epochs          = EPOCHS,
    batch_size      = BATCH_SIZE,
    validation_split= VAL_SPLIT,
    callbacks       = callbacks,
    verbose         = 1
)

print()
print('=' * 55)
print(f'  ✅ Entrenamiento finalizado')
print(f'  Épocas ejecutadas : {len(history.history["loss"])}')
print(f'  Mejor val_loss    : {min(history.history["val_loss"]):.6f}')
print('=' * 55)

## 📊 8. Evaluación y Métricas

In [ ]:
# ============================================================
# BLOQUE 8: EVALUACIÓN DEL MODELO Y MÉTRICAS
# ============================================================
# Se generan predicciones en escala original (desnormalizada)
# y se calculan métricas estándar de regresión.

# ── Predicciones en escala normalizada ──────────────────────
y_pred_train_norm = model.predict(X_train, verbose=0)
y_pred_test_norm  = model.predict(X_test,  verbose=0)

# ── Desnormalizar → valores reales de pasajeros ─────────────
y_pred_train = scaler.inverse_transform(y_pred_train_norm)
y_pred_test  = scaler.inverse_transform(y_pred_test_norm)
y_true_train = scaler.inverse_transform(y_train.reshape(-1, 1))
y_true_test  = scaler.inverse_transform(y_test.reshape(-1, 1))

# ── Cálculo de métricas ──────────────────────────────────────
def calcular_metricas(y_true, y_pred, nombre):
    """
    Calcula y muestra las métricas de evaluación:
    - RMSE : Raíz del error cuadrático medio (penaliza errores grandes)
    - MAE  : Error absoluto medio (interpretable en unidades originales)
    - MAPE : Error porcentual absoluto medio
    - R²   : Coeficiente de determinación (qué tan bien explica la varianza)
    """
    mse  = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae  = mean_absolute_error(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    r2   = r2_score(y_true, y_pred)

    print(f'  {nombre}')
    print(f'    RMSE : {rmse:.2f} pasajeros')
    print(f'    MAE  : {mae:.2f}  pasajeros')
    print(f'    MAPE : {mape:.2f} %')
    print(f'    R²   : {r2:.4f}')
    return {'RMSE': rmse, 'MAE': mae, 'MAPE': mape, 'R2': r2}

print('=' * 55)
print('   MÉTRICAS DE EVALUACIÓN')
print('=' * 55)
metricas_train = calcular_metricas(y_true_train, y_pred_train, '🔵 TRAIN')
print()
metricas_test  = calcular_metricas(y_true_test,  y_pred_test,  '🔴 TEST')
print('=' * 55)

## 📈 9. Visualización de Resultados

In [ ]:
# ============================================================
# BLOQUE 9A: CURVAS DE PÉRDIDA DURANTE EL ENTRENAMIENTO
# ============================================================
# Este gráfico permite detectar sobreajuste (overfitting):
# Si train_loss baja pero val_loss sube → el modelo está memorizando.

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Comportamiento del Entrenamiento LSTM',
             fontsize=15, fontweight='bold')

epochs_range = range(1, len(history.history['loss']) + 1)

# Pérdida (MSE)
axes[0].plot(epochs_range, history.history['loss'],
             color='#1565C0', linewidth=2, label='Train Loss (MSE)')
axes[0].plot(epochs_range, history.history['val_loss'],
             color='#C62828', linewidth=2, linestyle='--', label='Val Loss (MSE)')
axes[0].set_title('Función de Pérdida (MSE)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Época')
axes[0].set_ylabel('MSE')
axes[0].legend()
axes[0].set_yscale('log')   # Escala logarítmica para mejor visualización

# MAE
axes[1].plot(epochs_range, history.history['mae'],
             color='#2E7D32', linewidth=2, label='Train MAE')
axes[1].plot(epochs_range, history.history['val_mae'],
             color='#F57F17', linewidth=2, linestyle='--', label='Val MAE')
axes[1].set_title('Error Absoluto Medio (MAE)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Época')
axes[1].set_ylabel('MAE')
axes[1].legend()

plt.tight_layout()
plt.savefig('curvas_entrenamiento.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Curvas de entrenamiento generadas.')

In [ ]:
# ============================================================
# BLOQUE 9B: PREDICCIONES vs VALORES REALES
# ============================================================
# Comparación visual directa entre lo que el modelo predijo
# y los valores reales del dataset.

# ── Reconstruir el eje temporal ───────────────────────────────
fechas_total = df.index[LOOK_BACK:]
fechas_train = fechas_total[:train_size]
fechas_test  = fechas_total[train_size:]

fig, ax = plt.subplots(figsize=(15, 6))

# Serie original completa (referencia)
ax.plot(df.index, df['Pasajeros'],
        color='#B0BEC5', linewidth=1.5, alpha=0.6, label='Serie original')

# Predicciones de entrenamiento
ax.plot(fechas_train, y_pred_train.flatten(),
        color='#1E88E5', linewidth=2,
        linestyle='--', label='Predicción – Train')

# Predicciones de prueba
ax.plot(fechas_test, y_pred_test.flatten(),
        color='#E53935', linewidth=2.5, label='Predicción – Test')

# Valores reales de prueba
ax.plot(fechas_test, y_true_test.flatten(),
        color='#43A047', linewidth=2,
        linestyle=':', marker='o', markersize=4, label='Real – Test')

# Línea de corte
ax.axvline(fechas_test[0], color='gray',
           linestyle='--', linewidth=1.5, alpha=0.8, label='Inicio de prueba')
ax.fill_betweenx([df['Pasajeros'].min() - 20, df['Pasajeros'].max() + 20],
                  fechas_test[0], df.index[-1],
                  alpha=0.05, color='red')

ax.set_title('Predicciones LSTM vs Valores Reales – Airline Passengers',
             fontsize=14, fontweight='bold')
ax.set_ylabel('Número de Pasajeros (miles)')
ax.set_xlabel('Fecha')
ax.legend(loc='upper left', fontsize=10)

# Anotar métricas en el gráfico
texto = (f"RMSE: {metricas_test['RMSE']:.1f}\n"
         f"MAE:  {metricas_test['MAE']:.1f}\n"
         f"R²:   {metricas_test['R2']:.4f}")
ax.text(0.02, 0.97, texto, transform=ax.transAxes,
        verticalalignment='top', fontsize=10,
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

plt.tight_layout()
plt.savefig('predicciones_vs_reales.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Gráfico de predicciones generado.')

In [ ]:
# ============================================================
# BLOQUE 9C: SCATTER – PREDICHO vs REAL (sólo TEST)
# ============================================================
# Un buen modelo tendrá puntos cerca de la línea y = x (diagonal perfecta).

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Análisis de Errores del Modelo LSTM', fontsize=14, fontweight='bold')

# Gráfico de dispersión predicho vs real
axes[0].scatter(y_true_test, y_pred_test,
                color='#7B1FA2', alpha=0.8, edgecolors='white', s=80)
lim_min = min(y_true_test.min(), y_pred_test.min()) - 10
lim_max = max(y_true_test.max(), y_pred_test.max()) + 10
axes[0].plot([lim_min, lim_max], [lim_min, lim_max],
             'r--', linewidth=2, label='Predicción perfecta (y=x)')
axes[0].set_xlabel('Valores Reales (pasajeros)')
axes[0].set_ylabel('Valores Predichos (pasajeros)')
axes[0].set_title('Predicho vs Real – Conjunto de Prueba', fontweight='bold')
axes[0].legend()
axes[0].text(0.05, 0.95, f'R² = {metricas_test["R2"]:.4f}',
             transform=axes[0].transAxes, fontsize=11,
             verticalalignment='top',
             bbox=dict(boxstyle='round', facecolor='lavender'))

# Distribución de residuos
residuos = y_true_test.flatten() - y_pred_test.flatten()
axes[1].hist(residuos, bins=12, color='#00796B', edgecolor='white',
             alpha=0.85, linewidth=0.8)
axes[1].axvline(0, color='red', linestyle='--', linewidth=2, label='Error = 0')
axes[1].axvline(residuos.mean(), color='orange', linestyle='--',
                linewidth=2, label=f'Media = {residuos.mean():.1f}')
axes[1].set_xlabel('Residuo (Real – Predicho)')
axes[1].set_ylabel('Frecuencia')
axes[1].set_title('Distribución de Residuos', fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.savefig('analisis_errores.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Análisis de errores generado.')

In [ ]:
# ============================================================
# BLOQUE 9D: PREDICCIÓN FUTURA (PRONÓSTICO 12 MESES)
# ============================================================
# Se usa el modelo para predecir los 12 meses siguientes
# al período cubierto por el dataset (enero–diciembre 1961).

N_FUTURO = 12    # meses a pronosticar

# Última ventana de datos reales (ultimos 12 meses normalizados)
ultima_ventana = data_scaled[-LOOK_BACK:].reshape(1, LOOK_BACK, 1)

predicciones_futuras = []
ventana_actual = ultima_ventana.copy()

for _ in range(N_FUTURO):
    # Predecir el siguiente paso
    pred = model.predict(ventana_actual, verbose=0)[0, 0]
    predicciones_futuras.append(pred)
    # Desplazar la ventana: quitar el más antiguo, añadir la nueva predicción
    nueva_entrada = np.array([[[pred]]])
    ventana_actual = np.concatenate([ventana_actual[:, 1:, :], nueva_entrada], axis=1)

# Desnormalizar las predicciones
pred_futuras_reales = scaler.inverse_transform(
    np.array(predicciones_futuras).reshape(-1, 1)
)

# Crear fechas futuras (desde enero 1961)
ultima_fecha = df.index[-1]
fechas_futuras = pd.date_range(start=ultima_fecha + pd.DateOffset(months=1),
                               periods=N_FUTURO, freq='MS')

# ── Graficar el pronóstico ────────────────────────────────────
fig, ax = plt.subplots(figsize=(15, 6))

ax.plot(df.index, df['Pasajeros'],
        color='#1565C0', linewidth=2, label='Historia real (1949–1960)')
ax.plot(fechas_futuras, pred_futuras_reales.flatten(),
        color='#FF6F00', linewidth=2.5,
        linestyle='--', marker='o', markersize=6,
        label='Pronóstico LSTM (1961)')
ax.fill_between(fechas_futuras,
                pred_futuras_reales.flatten() * 0.92,
                pred_futuras_reales.flatten() * 1.08,
                alpha=0.15, color='#FF6F00', label='Intervalo ±8%')
ax.axvline(fechas_futuras[0], color='gray',
           linestyle='--', linewidth=1.5, alpha=0.7, label='Inicio del pronóstico')

ax.set_title('Pronóstico LSTM – 12 Meses Futuros (1961)',
             fontsize=14, fontweight='bold')
ax.set_ylabel('Número de Pasajeros (miles)')
ax.set_xlabel('Fecha')
ax.legend(loc='upper left', fontsize=10)

# Tabla de pronósticos
meses_nombres = ['Ene','Feb','Mar','Abr','May','Jun',
                 'Jul','Ago','Sep','Oct','Nov','Dic']
for i, (fecha, val) in enumerate(zip(fechas_futuras, pred_futuras_reales.flatten())):
    ax.annotate(f'{val:.0f}', (fecha, val),
                textcoords='offset points', xytext=(0, 10),
                fontsize=8, ha='center', color='#E65100')

plt.tight_layout()
plt.savefig('pronostico_futuro.png', dpi=150, bbox_inches='tight')
plt.show()

# Mostrar tabla de resultados futuros
df_futuro = pd.DataFrame({
    'Mes': [m.strftime('%B %Y') for m in fechas_futuras],
    'Pasajeros Pronosticados': pred_futuras_reales.flatten().astype(int)
})
print('=' * 45)
print('   PRONÓSTICO 1961 – 12 MESES FUTUROS')
print('=' * 45)
print(df_futuro.to_string(index=False))
print('=' * 45)

## 📋 10. Resumen Final y Conclusiones

> **¿Cómo usa la RNN/LSTM la información secuencial para predecir?**
>
> La LSTM procesa la ventana de 12 meses paso a paso.
> En cada paso `t` actualiza tres compuertas internas:
> - **Forget Gate**: decide qué información del pasado olvidar
> - **Input Gate**: decide qué nueva información guardar en la memoria de celda
> - **Output Gate**: decide qué parte de la memoria exponer como salida
>
> Al final de los 12 pasos, el vector de estado oculto `h₁₂`
> contiene un resumen comprimido de toda la historia,
> que la capa Dense transforma en la predicción del mes 13.

In [ ]:
# ============================================================
# BLOQUE 10: RESUMEN EJECUTIVO FINAL
# ============================================================

print()
print('╔' + '═'*53 + '╗')
print('║       RESUMEN EJECUTIVO – MODELO LSTM               ║')
print('╠' + '═'*53 + '╣')
print(f'║  Dataset        : Airline Passengers (1949–1960)    ║')
print(f'║  Arquitectura   : LSTM(128) → LSTM(64) → Dense(1)  ║')
print(f'║  Ventana (LB)   : {LOOK_BACK} meses                         ║')
print(f'║  Épocas         : {len(history.history["loss"]):<5}                        ║')
print('╠' + '═'*53 + '╣')
print('║              MÉTRICAS EN PRUEBA                     ║')
print('╠' + '═'*53 + '╣')
print(f'║  RMSE : {metricas_test["RMSE"]:>8.2f} pasajeros                   ║')
print(f'║  MAE  : {metricas_test["MAE"]:>8.2f} pasajeros                   ║')
print(f'║  MAPE : {metricas_test["MAPE"]:>8.2f} %                          ║')
print(f'║  R²   : {metricas_test["R2"]:>8.4f}                            ║')
print('╠' + '═'*53 + '╣')
print('║  Archivos generados:                                ║')
print('║   • exploracion_dataset.png                         ║')
print('║   • curvas_entrenamiento.png                        ║')
print('║   • predicciones_vs_reales.png                      ║')
print('║   • analisis_errores.png                            ║')
print('║   • pronostico_futuro.png                           ║')
print('║   • mejor_modelo_lstm.keras                         ║')
print('╚' + '═'*53 + '╝')
print()
print('✅ Notebook completado exitosamente.')